# Probe training on merged shards

Same logic as `train.ipynb`'s probe-training section. Only difference: `X` and `y` are loaded from the `merged shards (output of `shard_merge.ipynb`) instead of being produced in-memory by a generation loop.

Runs on CPU in a few minutes. No GPU, no compute units.

In [14]:
# === Colab bootstrap ===
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/cs639-outputs"
else:
    OUTPUT_DIR = "./outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"IN_COLAB={IN_COLAB}  OUTPUT_DIR={OUTPUT_DIR}")

IN_COLAB=False  OUTPUT_DIR=./outputs


In [15]:
%pip install -q torch pandas h5py scikit-learn optuna


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip3.14 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Config

In [16]:
# --- Inputs (from shard_merge.ipynb) ---
HDF5_PATH    = os.path.join(OUTPUT_DIR, "traces_all.h5")

# --- Outputs ---
PROBE_PATH   = os.path.join(OUTPUT_DIR, "linear_probe.pt")
METRICS_PATH = os.path.join(OUTPUT_DIR, "probe_metrics.json")

# --- Fixed ---
HIDDEN_DIM   = 3584   # DeepSeek-R1-Distill-Qwen-7B
SEED         = 0

# --- Split ratios (problem-level) ---
TRAIN_FRAC   = 0.70
VAL_FRAC     = 0.15
# TEST_FRAC implied: 1 - TRAIN_FRAC - VAL_FRAC

# --- Sweep config ---
N_TRIALS     = 40
# Sweep space (controlled in objective):
#   lr           : loguniform [1e-4, 1e-2]
#   weight_decay : loguniform [1e-6, 1e-1]
#   batch_size   : {128, 256, 512}
#   n_epochs     : int [20, 80]

In [17]:
# === Load X, y, pids from merged HDF5 ===
import torch
import h5py

assert os.path.exists(HDF5_PATH), f"Missing {HDF5_PATH} — run shard_merge.ipynb first"

X_parts, y_parts, pid_parts = [], [], []
with h5py.File(HDF5_PATH, "r") as f:
    for key in sorted(f.keys()):
        if not key.startswith("problem_"):
            continue
        grp = f[key]
        hs = torch.from_numpy(grp["hidden_states"][...]).float()
        if hs.shape[0] == 0:
            continue
        label = int(grp["label"][()])
        pid   = int(key.split("_", 1)[1])
        X_parts.append(hs)
        y_parts.extend([label] * hs.shape[0])
        pid_parts.extend([pid] * hs.shape[0])

X    = torch.cat(X_parts, dim=0)
y    = torch.tensor(y_parts,   dtype=torch.float32)
pids = torch.tensor(pid_parts, dtype=torch.long)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"X: {tuple(X.shape)}  y: {tuple(y.shape)}  pids: {tuple(pids.shape)}  device: {device}")
print(f"Unique problems: {pids.unique().numel()}")
print(f"Label distribution: {int(y.sum())} correct / {int((1 - y).sum())} incorrect")

X: (62985, 3584)  y: (62985,)  pids: (62985,)  device: cuda
Unique problems: 672
Label distribution: 34869 correct / 28116 incorrect


In [18]:
# === Group-aware 70/15/15 split by problem, then normalize (fit on train only) ===
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(SEED)

unique_pids = pids.unique()
shuffled    = unique_pids[torch.randperm(unique_pids.numel())]

n_problems = shuffled.numel()
n_train_p  = int(TRAIN_FRAC * n_problems)
n_val_p    = int(VAL_FRAC   * n_problems)
train_pids = shuffled[:n_train_p]
val_pids   = shuffled[n_train_p:n_train_p + n_val_p]
test_pids  = shuffled[n_train_p + n_val_p:]

def mask_of(target_pids):
    return torch.isin(pids, target_pids)

train_mask, val_mask, test_mask = mask_of(train_pids), mask_of(val_pids), mask_of(test_pids)

assert (train_mask & val_mask).sum() == 0
assert (train_mask & test_mask).sum() == 0
assert (val_mask & test_mask).sum() == 0
assert (train_mask | val_mask | test_mask).sum() == len(pids)

# Fit mean/std on train only, apply to all splits
mu  = X[train_mask].mean(0)
sig = X[train_mask].std(0).clamp(min=1e-6)
X   = (X - mu) / sig

train_ds = TensorDataset(X[train_mask], y[train_mask])
val_ds   = TensorDataset(X[val_mask],   y[val_mask])
test_ds  = TensorDataset(X[test_mask],  y[test_mask])

# train loader is created per trial (batch_size is a sweep param); val/test are fixed
val_loader  = DataLoader(val_ds,  batch_size=512)
test_loader = DataLoader(test_ds, batch_size=512)

def label_balance(yy):
    pos = int(yy.sum().item())
    return f"{pos}/{len(yy)} pos ({pos / max(len(yy), 1):.2%})"

print(f"Train: {len(train_pids)} problems, {int(train_mask.sum())} positions, {label_balance(y[train_mask])}")
print(f"Val:   {len(val_pids)} problems, {int(val_mask.sum())} positions, {label_balance(y[val_mask])}")
print(f"Test:  {len(test_pids)} problems, {int(test_mask.sum())} positions, {label_balance(y[test_mask])}")
print(f"Normalized X — train mean≈{X[train_mask].mean().item():.4f}  std≈{X[train_mask].std().item():.4f}")

Train: 470 problems, 44058 positions, 24447/44058 pos (55.49%)
Val:   100 problems, 9112 positions, 5607/9112 pos (61.53%)
Test:  102 problems, 9815 positions, 4815/9815 pos (49.06%)
Normalized X — train mean≈0.0000  std≈1.0000


In [19]:
# === Probe definition + evaluation helper ===
import torch.nn as nn
from sklearn.metrics import roc_auc_score

class LinearProbe(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        return self.fc(x).squeeze(-1)

    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))

def collect_probs(model, dl):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to(device)
            probs.append(torch.sigmoid(model(xb)).cpu())
            labels.append(yb)
    return torch.cat(probs), torch.cat(labels)

print(LinearProbe(HIDDEN_DIM))

LinearProbe(
  (fc): Linear(in_features=3584, out_features=1, bias=True)
)


In [20]:
# === Hyperparameter sweep (Optuna / TPE, 40 trials) ===
import optuna
from torch.utils.data import DataLoader

optuna.logging.set_verbosity(optuna.logging.WARNING)

best_val_auc = 0.0

def objective(trial):
    global best_val_auc

    lr           = trial.suggest_float("lr",           1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-1, log=True)
    batch_size   = trial.suggest_categorical("batch_size", [128, 256, 512])
    n_epochs     = trial.suggest_int("n_epochs", 20, 80)

    loader    = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    probe     = LinearProbe(HIDDEN_DIM).to(device)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    trial_best_auc   = 0.0
    trial_best_state = None

    for epoch in range(n_epochs):
        probe.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            criterion(probe(xb), yb).backward()
            optimizer.step()

        val_probs, val_labels = collect_probs(probe, val_loader)
        val_auc = roc_auc_score(val_labels.numpy(), val_probs.numpy())

        if val_auc > trial_best_auc:
            trial_best_auc   = val_auc
            trial_best_state = {k: v.cpu().clone() for k, v in probe.state_dict().items()}

        trial.report(val_auc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    if trial_best_auc > best_val_auc:
        best_val_auc = trial_best_auc
        torch.save(
            {
                "state_dict":   trial_best_state,
                "hidden_dim":   HIDDEN_DIM,
                "best_val_auc": best_val_auc,
                "norm_mean":    mu.cpu(),
                "norm_std":     sig.cpu(),
                "best_params":  dict(trial.params),
            },
            PROBE_PATH,
        )

    return trial_best_auc

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest val_auc = {best_val_auc:.4f}")
print(f"Best params  = {study.best_params}")

/home/avni/source/repos/cs639-final-project/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Best trial: 37. Best value: 0.870658: 100%|██████████| 40/40 [18:04<00:00, 27.11s/it]


Best val_auc = 0.8707
Best params  = {'lr': 0.00023039622594339337, 'weight_decay': 0.0021942316864640545, 'batch_size': 512, 'n_epochs': 51}


In [21]:
# === Test-set evaluation: reload best checkpoint, compute proposal metrics ===
import json
import numpy as np
from sklearn.metrics import (
    roc_auc_score, brier_score_loss,
    accuracy_score, precision_score, recall_score, f1_score,
)

ckpt = torch.load(PROBE_PATH, map_location=device)
best_probe = LinearProbe(ckpt["hidden_dim"]).to(device)
best_probe.load_state_dict(ckpt["state_dict"])
best_probe.eval()

test_probs, test_labels = collect_probs(best_probe, test_loader)
p = test_probs.numpy()
t = test_labels.numpy().astype(int)
preds = (p > 0.5).astype(int)

def expected_calibration_error(probs, labels, n_bins=15):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece, n = 0.0, len(probs)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        in_bin = (probs >= lo) & (probs < hi if i < n_bins - 1 else probs <= hi)
        if not in_bin.any():
            continue
        bin_conf = probs[in_bin].mean()
        bin_acc  = labels[in_bin].mean()
        ece += (in_bin.sum() / n) * abs(bin_conf - bin_acc)
    return float(ece)

metrics = {
    "roc_auc":           float(roc_auc_score(t, p)),
    "ece":               expected_calibration_error(p, t, n_bins=15),
    "brier":             float(brier_score_loss(t, p)),
    "accuracy":          float(accuracy_score(t, preds)),
    "precision":         float(precision_score(t, preds, zero_division=0)),
    "recall":            float(recall_score(t, preds, zero_division=0)),
    "macro_f1":          float(f1_score(t, preds, average="macro", zero_division=0)),
    "n_test_problems":   int(test_pids.numel()),
    "n_test_positions":  int(len(t)),
    "best_val_auc":      float(ckpt["best_val_auc"]),
    "best_params":       ckpt.get("best_params", {}),
}

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Best probe -> {PROBE_PATH}")
print(f"Metrics    -> {METRICS_PATH}")
for k, v in metrics.items():
    print(f"  {k:18s} {v}")

Best probe -> ./outputs/linear_probe.pt
Metrics    -> ./outputs/probe_metrics.json
  roc_auc            0.8107368639667704
  ece                0.0313460968347514
  brier              0.17843423783779144
  accuracy           0.7415180845644421
  precision          0.7390848026868178
  recall             0.7312564901349948
  macro_f1           0.7413685558074947
  n_test_problems    102
  n_test_positions   9815
  best_val_auc       0.8706581619114278
  best_params        {'lr': 0.00023039622594339337, 'weight_decay': 0.0021942316864640545, 'batch_size': 512, 'n_epochs': 51}


In [22]:
# === Smoke-check: reload checkpoint, verify predict_proba ∈ [0, 1], confirm norm params ===
fresh_ckpt  = torch.load(PROBE_PATH, map_location="cpu")
fresh_probe = LinearProbe(fresh_ckpt["hidden_dim"])
fresh_probe.load_state_dict(fresh_ckpt["state_dict"])
fresh_probe.eval()

assert "norm_mean" in fresh_ckpt and "norm_std" in fresh_ckpt, "norm params missing from checkpoint"

with torch.no_grad():
    sample = fresh_probe.predict_proba(X[:8])  # X is already normalized in this session

assert sample.dtype == torch.float32
assert (sample >= 0).all() and (sample <= 1).all(), f"out of range: {sample}"
print(f"predict_proba(X[:8]) = {sample.tolist()}")
print(f"Best params: {fresh_ckpt['best_params']}")
print("OK — checkpoint loads, predict_proba returns floats in [0, 1], norm params present.")

predict_proba(X[:8]) = [0.7348474264144897, 0.9956603646278381, 0.9463193416595459, 0.9371079206466675, 0.9283117055892944, 0.9312259554862976, 0.8960232138633728, 0.9169945120811462]
Best params: {'lr': 0.00023039622594339337, 'weight_decay': 0.0021942316864640545, 'batch_size': 512, 'n_epochs': 51}
OK — checkpoint loads, predict_proba returns floats in [0, 1], norm params present.
